# VITS 端到端语音合成教程

本教程介绍 VITS 端到端语音合成模型，包括：

1. **端到端 TTS 原理** - 直接从文本生成波形
2. **模型架构** - VAE + 流模型 + GAN
3. **核心组件** - 先验/后验编码器、流模型、解码器
4. **训练与推理** - 损失函数与生成过程
5. **实践应用** - 模型创建与语音生成

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 端到端 TTS 原理

### 1.1 传统两阶段 vs 端到端

**传统两阶段 TTS**:
```
文本 → [声学模型] → Mel频谱 → [声码器] → 波形
```

**VITS 端到端 TTS**:
```
文本 → [VITS] → 波形
```

### 1.2 VITS 的核心思想

VITS 结合了三种强大的生成模型：
- **VAE**: 学习潜在表示
- **流模型**: 学习复杂分布映射
- **GAN**: 生成高质量波形

In [ ]:
# 可视化 VITS 架构
fig, ax = plt.subplots(figsize=(14, 8))

# 绘制模块
modules = [
    ("文本", 0.5, 7, "lightblue"),
    ("先验编码器\n(文本→分布)", 0.5, 5.5, "lightgreen"),
    ("音频", 3.5, 7, "lightyellow"),
    ("后验编码器\n(音频→z)", 3.5, 5.5, "lightyellow"),
    ("流模型\n(分布映射)", 2, 4, "lightcoral"),
    ("时长预测器", 0.5, 3, "lightgray"),
    ("HiFi-GAN\n解码器", 2, 2, "plum"),
    ("波形", 2, 0.5, "lightblue"),
]

for name, x, y, color in modules:
    ax.add_patch(plt.Rectangle((x-0.4, y-0.4), 0.8, 0.8, 
                                fill=True, color=color, edgecolor='black'))
    ax.text(x, y, name, ha='center', va='center', fontsize=9)

# 绘制箭头
arrows = [
    (0.5, 6.6, 0.5, 5.9),
    (3.5, 6.6, 3.5, 5.9),
    (0.9, 5.5, 1.6, 4.4),
    (3.1, 5.5, 2.4, 4.4),
    (0.5, 5.1, 0.5, 3.4),
    (2, 3.6, 2, 2.4),
    (2, 1.6, 2, 0.9),
]
for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
               arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlim(-0.5, 4.5)
ax.set_ylim(0, 8)
ax.set_title('VITS 模型架构', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. VITS 模型架构

```
┌─────────────────────────────────────────────────────────────────┐
│                           VITS                                   │
├─────────────────────────────────────────────────────────────────┤
│  训练时:                                                         │
│  文本 → [先验编码器] → μ_p, σ_p (先验分布)                       │
│  音频 → [后验编码器] → μ_q, σ_q → z (潜在变量)                   │
│                              ↓                                   │
│                         [流模型]                                 │
│                              ↓                                   │
│                    z_p (变换后的潜在变量)                         │
│                              ↓                                   │
│                      [HiFi-GAN 解码器]                           │
│                              ↓                                   │
│                          重建音频                                │
├─────────────────────────────────────────────────────────────────┤
│  推理时:                                                         │
│  文本 → [先验编码器] → μ_p, σ_p                                  │
│                              ↓                                   │
│                    采样 z_p ~ N(μ_p, σ_p)                        │
│                              ↓                                   │
│                    [流模型逆变换]                                 │
│                              ↓                                   │
│                      [HiFi-GAN 解码器]                           │
│                              ↓                                   │
│                         生成音频                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
from vits import VITSConfig, VITS, create_vits_model

config = VITSConfig()
print("VITS 默认配置:")
print(f"  vocab_size: {config.vocab_size}")
print(f"  hidden_channels: {config.hidden_channels}")
print(f"  n_flows: {config.n_flows}")
print(f"  n_layers: {config.n_layers}")
print(f"  inter_channels: {config.inter_channels}")

### 2.1 先验编码器 (文本编码器)

将文本序列编码为潜在分布的参数 (μ, σ)。

In [ ]:
from vits import TextEncoder

text_encoder = TextEncoder(config)

batch_size = 2
text = torch.randint(0, config.vocab_size, (batch_size, 20))
text_lengths = torch.tensor([20, 15])

x, m_p, logs_p, x_mask = text_encoder(text, text_lengths)
print(f"输入文本: {text.shape}")
print(f"编码输出: {x.shape}")
print(f"先验均值 μ_p: {m_p.shape}")
print(f"先验对数方差 log(σ_p): {logs_p.shape}")

### 2.2 后验编码器

从音频频谱提取潜在表示 z ~ q(z|x)。

In [ ]:
from vits import PosteriorEncoder

posterior_encoder = PosteriorEncoder(config)

spec = torch.randn(batch_size, config.n_mels, 100)
spec_lengths = torch.tensor([100, 80])

z, m_q, logs_q, y_mask = posterior_encoder(spec, spec_lengths)
print(f"输入频谱: {spec.shape}")
print(f"潜在变量 z: {z.shape}")
print(f"后验均值 μ_q: {m_q.shape}")

### 2.3 流模型

流模型学习先验分布到后验分布的可逆映射。

In [ ]:
from vits import ResidualCouplingBlock

flow = ResidualCouplingBlock(config)

z_input = torch.randn(batch_size, config.inter_channels, 50)
mask = torch.ones(batch_size, 1, 50)

z_forward = flow(z_input, mask, reverse=False)
z_reverse = flow(z_forward, mask, reverse=True)

print(f"输入: {z_input.shape}")
print(f"正向变换: {z_forward.shape}")
print(f"逆向变换: {z_reverse.shape}")
print(f"重建误差: {(z_input - z_reverse).abs().mean().item():.6f}")

### 2.4 HiFi-GAN 解码器

将潜在表示解码为音频波形。

In [ ]:
from vits import Generator

decoder = Generator(config)

z = torch.randn(1, config.inter_channels, 50)
audio = decoder(z)

print(f"潜在变量: {z.shape}")
print(f"生成音频: {audio.shape}")
print(f"上采样率: {audio.shape[-1] / z.shape[-1]:.0f}x")

## 3. 训练与推理

### 3.1 损失函数

VITS 的训练损失包含：
- **重建损失**: 生成音频与真实音频的差异
- **KL 散度**: 后验分布与先验分布的差异
- **时长损失**: 预测时长与真实时长的差异
- **对抗损失**: GAN 判别器损失

In [ ]:
from vits import vits_loss, kl_divergence

# 模拟损失计算
audio_real = torch.randn(2, 1, 8192)
audio_fake = torch.randn(2, 1, 8192)
z_p = torch.randn(2, 192, 50)
m_p = torch.randn(2, 192, 50)
logs_p = torch.randn(2, 192, 50)
m_q = torch.randn(2, 192, 50)
logs_q = torch.randn(2, 192, 50)
y_mask = torch.ones(2, 1, 50)
duration_loss = torch.tensor(0.5)

total_loss, loss_dict = vits_loss(
    audio_real, audio_fake, z_p, m_p, logs_p, m_q, logs_q, y_mask, duration_loss
)

print("损失分解:")
for name, value in loss_dict.items():
    print(f"  {name}: {value.item():.4f}")

### 3.2 完整模型推理

In [ ]:
model = create_vits_model("tiny")
model.eval()

text = torch.randint(0, 100, (1, 20))
text_lengths = torch.tensor([20])

with torch.no_grad():
    audio = model.infer(text, text_lengths, noise_scale=0.667, length_scale=1.0)

print(f"输入文本: {text.shape}")
print(f"生成音频: {audio.shape}")

## 4. 实践应用

### 4.1 创建不同大小的模型

In [ ]:
for size in ["tiny", "base", "large"]:
    model = create_vits_model(size)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"{size:>6} 模型参数量: {num_params / 1e6:.2f}M")

## 总结

### VITS 的关键创新

1. **端到端**: 直接从文本生成波形，无需中间表示
2. **VAE + Flow**: 结合变分推断和流模型学习复杂分布
3. **随机时长**: 产生自然的韵律变化
4. **高质量**: GAN 训练产生高保真音频

### 与其他模型对比

| 特性 | FastSpeech2 + HiFi-GAN | VITS |
|------|------------------------|------|
| 架构 | 两阶段 | 端到端 |
| 中间表示 | Mel 频谱 | 潜在变量 |
| 训练 | 分开训练 | 联合训练 |
| 质量 | 高 | 更高 |